In [10]:
# Install packages
!pip install -qU langchain langchain-community langchain-core langchain-text-splitters langchain-groq sentence-transformers faiss-cpu

# Import libraries
from google.colab import userdata

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_groq import ChatGroq

# Load Groq API Key
api_key = userdata.get("GROQ_API_KEY")

# Create LLM
llm = ChatGroq(
    groq_api_key=api_key,
    model_name="openai/gpt-oss-120b"
)

# Sample Documents
documents = [

Document(page_content="""
Artificial Intelligence (AI) allows computers to perform tasks that usually require human intelligence.
AI is used in healthcare, banking, transportation and education.
"""),

Document(page_content="""
Machine Learning is a branch of Artificial Intelligence.
It learns patterns from data without explicit programming.
"""),

Document(page_content="""
Deep Learning uses neural networks.
It is useful for image recognition and speech recognition.
""")

]

# Split Documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = splitter.split_documents(documents)

# Embedding Model
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Vector Store
vectorstore = FAISS.from_documents(
    chunks,
    embedding
)

# User Query
query = "Where is Artificial Intelligence used?"

# Retrieve Documents
docs = vectorstore.similarity_search(
    query,
    k=2
)

context = "\n\n".join([doc.page_content for doc in docs])

# RAG Prompt
prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}

Answer:
"""

# RAG Response
rag_response = llm.invoke(prompt)

# Normal Response
normal_response = llm.invoke(query)

# Print Results
print("========== User Query ==========")
print(query)

print("\n========== Retrieved Context ==========")
print(context)

print("\n========== RAG Answer ==========")
print(rag_response.content)

print("\n========== Without Retrieval ==========")
print(normal_response.content)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

========== User Query ==========
Where is Artificial Intelligence used?

========== Retrieved Context ==========
Artificial Intelligence (AI) allows computers to perform tasks that usually require human intelligence.
AI is used in healthcare, banking, transportation and education.

Machine Learning is a branch of Artificial Intelligence.
It learns patterns from data without explicit programming.

========== RAG Answer ==========
Artificial Intelligence is used in **healthcare, banking, transportation, and education**.

========== Without Retrieval ==========
**Artificial Intelligence (AI) is now a core technology in virtually every industry.** Below is a high‑level map of where AI is used today, organized by sector and by the type of AI techniques that make it possible.

---

## 1. Healthcare & Life Sciences
| AI Technique | Typical Applications | Real‑World Examples |
|--------------|----------------------|----------------------|
| **Machine Learning (ML) / Deep Learning** | Disease d